Train the network with no CBF. Note that this is just a test to validate that all the dataset generation works properly. This also tests that if we don't provide the directories for the keep-out (KO) regions then the dataset when used in the loader will only output the required tensors without the currently enabled KO.

In [24]:
import torch
import torch.nn as nn
import tqdm
import pathlib

from enum import Enum
from dataclasses import dataclass
from datetime import datetime
from torch.utils.data import DataLoader

from controller import Controller
from diff_mfld_optim.optim.subsolver import OptimFunc, FuncArgs
from diff_mfld_optim.mfld_util import MfldCfg, dist_squared_map

from geo_diff_opt_layer_ml.util.nominal_mpc_dataloader import (
    EPISODES_TRAIN_DIR,
    EPISODES_VALID_DIR,
    MPC_TRAIN_DIR,
    MPC_VALID_DIR,
    MPCEpisode,
    MPCEpisodeDataset,
)

In [ ]:
# device = torch.device(
#     "cuda:0"
#     if torch.cuda.is_available()
#     else "mps" if torch.backends.mps.is_available() else "cpu"
# )
device = torch.device("cpu")
device

In [2]:
mpc_train_dataset = MPCEpisodeDataset(EPISODES_TRAIN_DIR, MPC_TRAIN_DIR, device=device)
mpc_valid_dataset = MPCEpisodeDataset(EPISODES_VALID_DIR, MPC_VALID_DIR, device=device)

/home/samuel/Documents/School/UWaterloo_MASc/Research_Activities/research-geo-diff-opt-layer/geo_diff_opt_layer_ml/util/nominal_mpc_dataloader.py:131: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  torch.tensor(mpc_f["p_hist"], dtype=torch.float32, device=device),


In [4]:
def train_loop_no_safety(dataloader, model, loss_fn, optimizer):
    model.train()

    batch_loss = []
    for _batch, (p, x_traj, y_traj, u) in enumerate(dataloader):
        # makes a prediction on the control inputs
        pred_u = model(p, x_traj, y_traj)
        loss = loss_fn(pred_u, u)

        # backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        batch_loss.append(loss.item())

    # this will be our metric of performance
    avg_batch_loss = sum(batch_loss) / len(batch_loss)
    return avg_batch_loss


def valid_loop_no_safety(dataloader, model, loss_fn):
    model.eval()

    batch_loss = []
    for _batch, (p, x_traj, y_traj, u) in enumerate(dataloader):
        # makes a prediction on the control inputs
        pred_u = model(p, x_traj, y_traj)
        loss = loss_fn(pred_u, u)

        batch_loss.append(loss.item())

    # this will be our metric of performance
    avg_batch_loss = sum(batch_loss) / len(batch_loss)
    return avg_batch_loss

In [10]:
# controller params
state_dim = 5  # num states of dynamic unicycle
control_dim = 2  # num of inputs
traj_dim = 2  # only feeding in the trajectory position (not vel., accel.)

num_hidden_1 = 60
num_hidden_2 = 60

cntrllr_model = Controller(
    state_dim=state_dim,
    control_dim=control_dim,
    traj_dim=traj_dim,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    has_cbfs=False,
).to(device)

In [ ]:
# hyperparameters
epochs = 50
batch_size = 32
lr = 0.001

mpc_train_loader = DataLoader(mpc_train_dataset, batch_size=batch_size, shuffle=True)
mpc_valid_loader = DataLoader(mpc_valid_dataset, batch_size=batch_size, shuffle=True)


loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(params=cntrllr_model.parameters(), lr=lr)

# train the model

pbar = tqdm.tqdm(range(epochs), desc="Training")
for epoch in pbar:
    train_loss = train_loop_no_safety(
        mpc_train_loader, cntrllr_model, loss_fn, optimizer
    )
    valid_loss = valid_loop_no_safety(mpc_valid_loader, cntrllr_model, loss_fn)
    pbar.set_postfix(train_loss=train_loss, valid_loss=valid_loss)

Training: 100%|██████████| 50/50 [00:54<00:00,  1.10s/it, train_loss=0.0153, valid_loss=0.00735]


In [36]:
# save the model with a unique timestamp (to prevent overwriting models)
model_path = pathlib.Path("nn_no_cbf").joinpath(
    f"nn_no_cbf_{datetime.now().strftime("%Y_%m_%d__%H_%M")}.pth"
)
torch.save(cntrllr_model.state_dict(), model_path)